# Week 5 · 扩规模 + 加宋诗宋词

> **本周一句话**:数据从 4M → 22M 字符,模型从 6.37M → 25M 参数,vocab 从 9k → 14k —— val loss 跌到 4.03,生成开始出现真正的"金句"。

这是**数据 + 模型双管齐下**的一周。单独扩任意一个收益都有限,两个一起上才有质变。

## 0. 本周目标

| 维度 | v0.7 (上周) | v0.8 (本周) | 倍数 |
|---|---|---|---|
| 数据字符数 | 4 M(仅唐诗) | 22 M(唐+宋诗+宋词) | 5.5× |
| vocab_size | 9,563 | ~14,500 | 1.5× |
| n_embed | 192 | 384 | 2× |
| n_layer | 6 | 8 | 1.3× |
| block_size | 128 | 256 | 2× |
| 参数量 | 6.37 M | ~25 M | 3.9× |
| val loss | 4.21 | **~4.03** | -0.18 |
| 生成质量 | 押韵、句式 | + 用词典雅、意境连贯 | 质变 |

## 1. 前置知识

**必备**:
- Week 4 跑完(checkpoint 工程通了)
- 知道 GPU 显存怎么算(`参数 + 优化器状态 + activation`)

**这周第一次遇到**:
- 多数据源合并 + 打乱(为什么必须打乱)
- vocab 扩展的兼容性
- 大模型上 batch_size 的取舍
- `.clone()` 切断 view 共享(否则 .pt 文件翻倍)

## 2. 核心概念

### 2.1 为什么宋诗 + 宋词必须打乱

`prepare_v2.py` 的关键一行:

```python
random.seed(42)
random.shuffle(poems)        # ← 这行!
```

如果不打乱,数据顺序是:

```
  [唐诗 5w 首] → [宋诗 26w 首] → [宋词 2w 首]
```

90/10 切分之后,**验证集前 90% 是宋词、后 10% 是宋词 + 末尾的少量** —— 模型完全没在唐诗 / 宋诗上被验证过。val loss 会变得"看上去很好"但不真实。

打乱之后,train 和 val 都是三种来源混合采样,验证才有意义。

**通用原则**:多源数据 → 必须打乱再切。这个 bug 在大规模训练里很常见且很隐蔽。

In [ ]:
# 演示打乱前后验证集的"成分差异"
import random

poems = ["[T]"] * 5000 + ["[S诗]"] * 26000 + ["[S词]"] * 2000

def val_composition(p):
    n = int(0.95 * len(p))
    val = p[n:]
    return {x: val.count(x) for x in set(val)}

print("打乱前 val:", val_composition(poems))
random.seed(42); random.shuffle(poems)
print("打乱后 val:", val_composition(poems))

### 2.2 vocab 扩展的兼容性问题

宋诗宋词带来一些唐诗里没有的字(冷僻字、词牌专用字)。重新统计后 vocab 从 9563 涨到 ~14500。

**关键决策**:tokenizer_v1 和 tokenizer_v2 是**完全独立**的两个文件,**ID 编号不兼容**。

```
v1: stoi["月"] = 4287
v2: stoi["月"] = 6512    # ← 重排了
```

所以 v0.8 不能直接 load v0.7 的权重 —— embedding 的"行号意义"对不上。Week 5 干脆**从头训** v0.8,放弃 v0.7 的权重(数据和模型都变了,迁移成本不如重训)。

**Week 6 SFT 时**,vocab 只加了 5 个特殊 token(append 到末尾),旧 ID 不动,这时候才有"权重迁移"的可能。

### 2.3 大模型的显存账

25M 模型 + AMP + block_size=256 + batch_size=32 在 T4(15.6 GB)能不能跑?粗算:

| 项目 | 估算公式 | 数值 |
|---|---|---|
| 模型 fp16 权重 | 25M × 2 bytes | 50 MB |
| 模型 fp32 master | 25M × 4 bytes | 100 MB |
| AdamW m + v 状态 | 25M × 2 × 4 bytes | 200 MB |
| Activation(主要)| `B × T × C × n_layer × ~5` | 32 × 256 × 384 × 8 × 5 ≈ 130 MB |
| Attention scores | `B × n_head × T²` | 32 × 6 × 256² ≈ 12 MB |
| **总计** | | **~500 MB** |

T4 富富有余。但 batch_size 调到 64 + block_size 调到 512 就接近上限了。

**实际 monitor**:训练前 `nvidia-smi` 看 baseline,训练第 100 步再看,差值就是占用。

### 2.4 .clone() 切断 view 共享

一个隐蔽的内存陷阱:

```python
data = torch.tensor(...)           # 22 MB
train = data[:int(0.95*len(data))]    # view,共享 storage
val   = data[int(0.95*len(data)):]    # view,共享 storage

torch.save(train, "train.pt")      # ← 实际写入 22 MB(整个 storage),不是 21 MB!
torch.save(val,   "val.pt")        # ← 又写入 22 MB!
```

`torch.save` 看到的是底层 storage,view 共享同一份 storage 就各自序列化整份 → 文件大小翻倍。

**修复**:存之前 `.clone()`,创建独立 storage:

```python
train = data[:n].clone()
val   = data[n:].clone()
torch.save(train, ...)             # 现在只写 21 MB
torch.save(val,   ...)             # 只写 1 MB
```

`prepare_v2.py` 里就加了这一行,**别去掉**。

## 3. 代码地图

| 文件 | 行数 | 干什么 |
|---|---|---|
| `data/prepare_v2.py` | 100 | 唐+宋诗+宋词合并,打乱,清洗,建 v2 tokenizer,切分 |
| `configs/config.py:MiniGPTv08Cfg` | 6 | n_embed=384, n_head=6, n_layer=8, block_size=256 |
| `configs/config.py:TrainV08Cfg` | 9 | num_steps=10000, batch=32, max_lr=3e-4 |
| `train/train_v08.py` | 165 | 用现有 `MiniGPTv03` 类 + v2 数据 + 全套工程化 |

**注意**:v0.8 没有新的 model 类,**直接复用 v0.3 的 `MiniGPTv03`,只改超参**。架构演进到 v0.3 就已经定型了,后面 5/6/7 都靠它。这是好的设计:模型类不变,通过 config 切尺寸。

## 4. 动手做

In [ ]:
import subprocess, sys
from pathlib import Path


def _find_repo_root() -> Path:
    """定位仓库根目录(含 pyproject.toml + train/),不依赖 notebook 工作目录。
    - 本地/Colab:cwd 在仓库内,从 cwd 往上找即命中。
    - 魔搭 ModelScope:kernel cwd 在 /mnt/workspace,但仓库克隆在 home,
      所以再去 home / 常见根目录下搜几层。"""
    def ok(d):
        return (d / "pyproject.toml").is_file() and (d / "train").is_dir()
    here = Path.cwd().resolve()
    for d in (here, *here.parents):
        if ok(d):
            return d
    bases, seen = [here, Path.home(), Path("/home"), Path("/root"), Path("/mnt")], set()
    for base in bases:
        try:
            base = base.resolve()
        except Exception:
            continue
        if not base.is_dir() or base in seen:
            continue
        seen.add(base)
        for depth in ("*", "*/*", "*/*/*"):
            for f in base.glob(f"{depth}/pyproject.toml"):
                if (f.parent / "train").is_dir():
                    return f.parent.resolve()
    raise RuntimeError(f"找不到仓库根(应含 pyproject.toml + train/),cwd={here},home={Path.home()}")


REPO = _find_repo_root()                      # 项目根的绝对路径
print(f"REPO = {REPO}")


def run(cmd):
    """跑子进程并把输出实时打印到 cell。
    - "python" 换成 sys.executable,确保用当前 kernel 解释器。
    - 形如 "../train/x.py" 的相对路径统一解析成 REPO 下的绝对路径,
      不再依赖 notebook 的工作目录(魔搭与本地/Colab 的 cwd 不一致)。
    - cwd=REPO 兜底:即便脚本内部用了相对路径也能找到文件。
    - Popen 逐行回读才能在 cell 里实时看到脚本的 print。"""
    cmd = [sys.executable if c == "python" else c for c in cmd]
    cmd = [str(REPO / c.removeprefix("../")) if isinstance(c, str) and c.startswith("../") else c
           for c in cmd]
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", bufsize=1, cwd=str(REPO),
    )
    for line in proc.stdout:
        print(line, end="")
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(f"{cmd} 退出码 {proc.returncode}")

# 先准备 v2 数据(下载宋诗宋词 + 合并 + 清洗 + 编码)
# 注意: chinese-poetry 仓库已经在 Week 1 clone 过,这里只下载新分支
run(["python", "../data/prepare_v2.py"])
# 预期:
#   唐诗: 57603, 宋诗: 286986, 宋词: 19990   (总 ~36 万首)
#   清洗后约 22M 字符
#   vocab_size: ~14500
#   train: ~21M tokens, val: ~1M tokens

In [ ]:
# 训练 v0.8 (T4 大约 40 分钟,可以喝杯咖啡)
run(["python", "../train/train_v08.py"])
# 预期: val loss 跌到 ~4.03
# best.pt 和 latest.pt 各约 300 MB(25M 模型 + 优化器 + scaler)

In [ ]:
# 看一下 v0.8 写诗
# (注意 train_v08 没有专门的 generate 入口,可以借用 generate.py 的逻辑)
import torch, pickle
import sys
sys.path.insert(0, str(REPO))                 # REPO 在上面 run() 的 cell 已定位
from model.minigpt_v03 import MiniGPTv03
from tokenizer.tokenizer import CharTokenizer
from configs.config import CHECKPOINT_DIR, TOKENIZER_V2_FILE

device = "cuda" if torch.cuda.is_available() else "cpu"
ckpt = torch.load(CHECKPOINT_DIR / "v08" / "best.pt", map_location=device, weights_only=False)
cfg = ckpt["config"]
model = MiniGPTv03(cfg["vocab_size"], cfg["n_embed"], cfg["n_head"],
                   cfg["n_layer"], cfg["block_size"], cfg["dropout"]).to(device)
model.load_state_dict(ckpt["model_state"])

tok = CharTokenizer.load(TOKENIZER_V2_FILE)
start = torch.tensor([tok.encode("月")], dtype=torch.long, device=device)
out = model.generate(start, max_new_tokens=200, temperature=0.8)
print(tok.decode(out[0].tolist()))

## 5. 自测题

**A. 数据**
- A1 唐诗 5w + 宋诗 28w + 宋词 2w,为什么宋诗数量最大?如果只想训写"五言绝句",这种分布合适吗?
- A2 不打乱直接切的话,val 会全是哪种?对训练时 val 曲线的判断会有什么误导?
- A3 v2 切 95/5 而不是 v1 的 90/10,理由是什么?

**B. vocab**
- B1 v1 → v2 vocab 涨了 ~5000 字,但模型参数量没涨那么多 —— 多出来的字主要影响哪几层?
- B2 如果 v0.8 训完之后,某天突然想加 v3 的"明清诗",怎么处理 vocab 兼容?
- B3 字符级 tokenizer 在 vocab 涨到 5w(全字符全标点)时会怎样?

**C. 模型规模**
- C1 n_embed 从 192 涨到 384,参数量为什么涨了 3.9× 而不是 2×?
- C2 block_size 从 128 涨到 256,attention 计算量涨了多少倍?
- C3 25M 模型在 T4 上 batch_size 极限大概是多少?(自己用上面的显存公式估)

**D. 训练**
- D1 v0.8 训 10000 步,为什么 warmup 从 v0.5 的 200 涨到 500?
- D2 val 4.03 vs v0.7 的 4.21,看着只降了 0.18,生成质量真的会有"质变"吗?为什么?
- D3 如果不重新训,直接用 v0.5 的权重 + v2 数据继续 fine-tune 会怎样?

## 6. 容易踩的坑

**坑 1:`tang_poems_clean.txt` 还在,误用了 v1 的清洗后文件**

v1 和 v2 的清洗后文件分别叫 `tang_poems_clean.txt` 和 `poems_v2_clean.txt`,**别混了**。v2 train 时如果误读 v1 文本,vocab 不一致 → 编码崩。

**坑 2:25M 模型 + block_size=256,batch_size=64 会 OOM**

attention scores 是 `B × n_head × T²`,T=256 时一个 batch 32 already 12 MB,翻倍就是 24 MB,反向传播时再 ×3 ~ ×4 (中间 activation)。T4 上 batch=32 安全,64 风险大。

**坑 3:中途 OOM 之后没清缓存,后续训练莫名其妙慢**

`torch.cuda.empty_cache()` 不是必须,但 OOM 后建议跑一次。或者干脆 restart kernel。

**坑 4:`prepare_v2.py` 跑到一半网络断,chinese-poetry clone 失败**

`data/download.py` 是 idempotent 的(检查到目录存在就跳过)。但如果上次 clone 到一半留下一个空目录或损坏目录,手动 `rm -rf raw/chinese-poetry/` 重来。

**坑 5:训练 30 分钟之后笔记本休眠 → 训练中断 → 没用 checkpoint 全废**

Week 4 的 checkpoint 系统就是为这种情况准备的。`train_v08.py` 每 1000 步存 latest,休眠回来后 `python train/train_v07.py --resume` 改成 v08 版续训。

## 7. 进入 Week 6 前

现在你应该:
- ☑ `data/processed/` 下既有 v1 文件又有 v2 文件,大小约 100 MB
- ☑ `checkpoints/v08/best.pt` 存在,~100 MB
- ☑ v0.8 生成的诗用词比 v0.7 明显更典雅,押韵更稳
- ☑ 能解释为什么 v1 → v2 必须重新训而不能 fine-tune

Week 6 我们要给 v0.8 加"指令能力" —— 不再是"给一个字续写",而是"给我题目 + 风格,你按要求写一首"。需要 SFT(Supervised Fine-Tuning) + 特殊 token + prompt mask。